# Feature Importance — Mental Health Monitoring

Predict a composite **distress score** (average of normalised PHQ-8 and PCL-C) from all acoustic and language features.

**Target construction**
- PHQ-8 (`Depression_severity`, 0–24) → divide by 24 → 0–1
- PCL-C (`PTSD_severity`, 0–68) → divide by 68 → 0–1
- `composite_score = (phq8_norm + pclc_norm) / 2`

> Note: The dataset does not include a Perceived Stress Scale (PSS) instrument.  
> The PCL-C (PTSD Checklist – Civilian) is used as the stress/trauma measure alongside PHQ-8.

---
### Sections
1. Load & merge data  
2. Exploratory data analysis  
3. Preprocessing  
4. Baseline: Random Forest  
5. Feature importance (RF + permutation)  
6. Model comparison  
7. Classification variant (binary risk)  
8. Export artefacts  

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install pandas numpy scikit-learn xgboost matplotlib seaborn shap

import warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Scikit-learn
from sklearn.ensemble          import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model      import Ridge, Lasso, ElasticNet
from sklearn.svm               import SVR
from sklearn.preprocessing     import StandardScaler
from sklearn.pipeline          import Pipeline
from sklearn.model_selection   import cross_validate, KFold, GridSearchCV
from sklearn.inspection        import permutation_importance
from sklearn.metrics           import mean_squared_error, mean_absolute_error, r2_score

# XGBoost (optional, comment out if not installed)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed – skipping XGB model')

# SHAP (optional)
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('SHAP not installed – skipping SHAP plots')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('Setup complete.')

## 1  Load & merge data

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = Path('.')  # notebook lives in the worktree root
FEATURES_CSV = ROOT / '../../daic_features_v4.csv'          # repo root
LABELS_CSV   = ROOT / 'data/edaic/labels/detailed_labels.csv'

# Fallback: try features.csv in same directory if daic_features_v4 not found
if not FEATURES_CSV.exists():
    FEATURES_CSV = ROOT / 'data/edaic/features/features.csv'

print('Features :', FEATURES_CSV.resolve())
print('Labels   :', LABELS_CSV.resolve())
print('Exists   :', FEATURES_CSV.exists(), LABELS_CSV.exists())

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
feat_df = pd.read_csv(FEATURES_CSV)
lab_df  = pd.read_csv(LABELS_CSV)

# Normalise participant ID column name
feat_id_col = 'patient_id'    if 'patient_id'      in feat_df.columns else 'participant_id'
lab_id_col  = 'Participant'   if 'Participant'      in lab_df.columns  else 'participant_id'

# Coerce to int for reliable join
feat_df[feat_id_col] = feat_df[feat_id_col].apply(lambda x: int(float(x)))
lab_df [lab_id_col]  = lab_df [lab_id_col].apply( lambda x: int(float(x)))

# Merge
df = feat_df.merge(
    lab_df[[lab_id_col, 'Depression_severity', 'PTSD_severity',
            'Depression_label', 'PTSD_label', 'gender', 'age', 'split']],
    left_on=feat_id_col, right_on=lab_id_col, how='inner'
)

print(f'Features : {feat_df.shape}  |  Labels : {lab_df.shape}  |  Merged : {df.shape}')

In [ ]:
# ── Build composite target ────────────────────────────────────────────────────
PHQ8_MAX = 24.0
PCLC_MAX = 68.0

df['phq8_norm']       = df['Depression_severity'].astype(float) / PHQ8_MAX
df['pclc_norm']       = df['PTSD_severity'].astype(float)       / PCLC_MAX
df['composite_score'] = (df['phq8_norm'] + df['pclc_norm']) / 2.0

print('Target distribution:')
print(df['composite_score'].describe().round(3))

# Binary risk label: composite_score >= 0.3 → high risk
THRESHOLD = 0.30
df['risk_label'] = (df['composite_score'] >= THRESHOLD).astype(int)
print(f'\nRisk label (threshold={THRESHOLD}) — class balance:')
print(df['risk_label'].value_counts())

## 2  Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df['Depression_severity'], bins=15, color='#5F9B6B', edgecolor='white')
axes[0].set_title('PHQ-8 Distribution'); axes[0].set_xlabel('PHQ-8 score (0–24)')

axes[1].hist(df['PTSD_severity'], bins=15, color='#4A7DBF', edgecolor='white')
axes[1].set_title('PCL-C Distribution'); axes[1].set_xlabel('PCL-C score (0–68)')

axes[2].hist(df['composite_score'], bins=15, color='#C4715A', edgecolor='white')
axes[2].set_title('Composite Score Distribution')
axes[2].set_xlabel('Composite (0–1)')
axes[2].axvline(THRESHOLD, color='black', ls='--', label=f'Threshold {THRESHOLD}')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Missing values
feat_cols_all = [c for c in feat_df.columns if c != feat_id_col]

missing = df[feat_cols_all].isnull().mean().sort_values(ascending=False)
print(f'Features with >10% missing: {(missing > 0.1).sum()}')
if missing.max() > 0:
    print(missing[missing > 0].to_string())
else:
    print('No missing values in features.')

In [ ]:
# Top correlations with composite_score
corr = df[feat_cols_all + ['composite_score']].corr()['composite_score'].drop('composite_score')
top_corr = corr.abs().sort_values(ascending=False).head(20)

plt.figure(figsize=(9, 5))
colors = ['#5F9B6B' if corr[i] > 0 else '#C4715A' for i in top_corr.index]
plt.barh(top_corr.index[::-1], top_corr.values[::-1], color=colors[::-1])
plt.xlabel('|Pearson r| with composite_score')
plt.title('Top 20 features by correlation with composite score')
plt.tight_layout()
plt.show()

print('\nTop 10 positively correlated:')
print(corr.sort_values(ascending=False).head(10).round(3))
print('\nTop 10 negatively correlated:')
print(corr.sort_values().head(10).round(3))

## 2b  Risk score vs PHQ-8 analysis

Explore how the composite risk score relates to the raw PHQ-8 depression score:
- Correlation (Pearson & Spearman)
- Scatter plots with regression line
- PHQ-8 severity band breakdown
- Agreement between risk label and clinical PHQ-8 cutoff (≥10 = depression)

In [ ]:
# ── PHQ-8 sub-items vs composite score ───────────────────────────────────────
# Check which individual PHQ-8 items drive the composite most
phq_items = [c for c in df.columns if c.startswith('PHQ8_') or c.startswith('PHQ_8')]
if phq_items:
    item_corr = df[phq_items + ['composite_score']].corr()['composite_score']\
                  .drop('composite_score').sort_values(ascending=False)
    
    # Clean up labels
    clean_labels = [c.replace('PHQ8_','').replace('PHQ_8','').replace('_',' ') for c in item_corr.index]
    
    plt.figure(figsize=(8, 4))
    colors_items = ['#C4715A' if v > 0 else '#4A7DBF' for v in item_corr.values]
    plt.barh(clean_labels[::-1], item_corr.values[::-1], color=colors_items[::-1])
    plt.xlabel('Pearson r with composite_score')
    plt.title('PHQ-8 sub-item correlation with composite risk score')
    plt.axvline(0, color='black', lw=0.5)
    plt.tight_layout()
    plt.show()
    print(item_corr.round(3).to_string())
else:
    # PHQ-8 sub-items not in merged df — load from labels CSV
    lab_full = pd.read_csv(LABELS_CSV)
    phq_items = [c for c in lab_full.columns if 'PHQ' in c and c != 'Depression_severity']
    lab_id    = 'Participant' if 'Participant' in lab_full.columns else 'participant_id'
    lab_full[lab_id] = lab_full[lab_id].apply(lambda x: int(float(x)))
    merged_phq = lab_full[[lab_id] + phq_items].merge(
        df[[feat_id_col, 'composite_score']], left_on=lab_id, right_on=feat_id_col
    )
    item_corr = merged_phq[phq_items + ['composite_score']].corr()['composite_score']\
                  .drop('composite_score').sort_values(ascending=False)
    clean_labels = [c.replace('PHQ8_','').replace('PHQ_8','').replace('_',' ') for c in item_corr.index]
    plt.figure(figsize=(8, 4))
    colors_items = ['#C4715A' if v > 0 else '#4A7DBF' for v in item_corr.values]
    plt.barh(clean_labels[::-1], item_corr.values[::-1], color=colors_items[::-1])
    plt.xlabel('Pearson r with composite_score')
    plt.title('PHQ-8 sub-item correlation with composite risk score')
    plt.axvline(0, color='black', lw=0.5)
    plt.tight_layout()
    plt.show()
    print(item_corr.round(3).to_string())

In [ ]:
# ── Agreement: risk_label vs clinical PHQ-8 depression cutoff ────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# PHQ-8 ≥ 10 is the standard clinical cutoff for depression
df['phq8_depressed'] = (df['Depression_severity'].astype(float) >= 10).astype(int)

agree  = (df['risk_label'] == df['phq8_depressed']).mean()
kappa  = stats.kendalltau(df['risk_label'], df['phq8_depressed']).statistic

print(f'Agreement between composite risk label and PHQ-8 ≥ 10: {agree:.2%}')
print(f"Kendall's τ : {kappa:.4f}")

ct = pd.crosstab(df['risk_label'], df['phq8_depressed'],
                 rownames=['Composite risk label'], colnames=['PHQ-8 ≥ 10'])
print('\nCross-tabulation:')
print(ct)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(df['phq8_depressed'], df['risk_label'])
disp = ConfusionMatrixDisplay(cm,
        display_labels=['PHQ-8 < 10\n(Not depressed)', 'PHQ-8 ≥ 10\n(Depressed)'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('PHQ-8 cutoff  vs  Composite risk label\n(rows=PHQ-8, cols=composite)')
axes[0].set_xlabel('Composite risk label (0=low, 1=high)')

# Stacked bar: among each PHQ-8 band, fraction flagged as high risk
band_risk = df.groupby('phq8_band', observed=True)['risk_label'].value_counts(normalize=True).unstack().fillna(0)
band_risk.columns = ['Low risk','High risk']
band_risk[['Low risk','High risk']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#5F9B6B','#C4715A'], edgecolor='white'
)
axes[1].set_xlabel('PHQ-8 severity band')
axes[1].set_ylabel('Fraction of participants')
axes[1].set_title('High-risk rate per PHQ-8 severity band')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── PHQ-8 severity bands ──────────────────────────────────────────────────────
# Clinical PHQ-8 cutoffs: 0-4 Minimal, 5-9 Mild, 10-14 Moderate,
#                          15-19 Moderately severe, 20-24 Severe
bins   = [-1, 4, 9, 14, 19, 24]
labels = ['Minimal\n(0–4)', 'Mild\n(5–9)', 'Moderate\n(10–14)',
          'Mod-Severe\n(15–19)', 'Severe\n(20–24)']
df['phq8_band'] = pd.cut(df['Depression_severity'].astype(float),
                          bins=bins, labels=labels)

band_counts = df['phq8_band'].value_counts().sort_index()
print('Participant count per PHQ-8 severity band:')
print(band_counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot: composite score per band
band_order = labels
plot_data  = [df[df['phq8_band'] == b]['composite_score'].dropna().values
              for b in band_order]
bp = axes[0].boxplot(plot_data, labels=band_order, patch_artist=True,
                     medianprops=dict(color='white', lw=2))
band_colors = ['#A8C4AF','#7DB08A','#5F9B6B','#C4715A','#8B2500']
for patch, color in zip(bp['boxes'], band_colors):
    patch.set_facecolor(color)
axes[0].set_ylabel('Composite risk score')
axes[0].set_title('Composite score distribution by PHQ-8 severity band')
axes[0].axhline(THRESHOLD, color='grey', ls='--', lw=1, label=f'Risk threshold {THRESHOLD}')
axes[0].legend(fontsize=8)

# Bar: mean composite score per band with 95% CI
means  = [np.mean(d) for d in plot_data]
errors = [1.96 * np.std(d) / np.sqrt(len(d)) if len(d) > 1 else 0 for d in plot_data]
axes[1].bar(band_order, means, color=band_colors, edgecolor='white', yerr=errors, capsize=4)
axes[1].set_ylabel('Mean composite risk score')
axes[1].set_title('Mean composite score per PHQ-8 band (± 95% CI)')
axes[1].axhline(THRESHOLD, color='grey', ls='--', lw=1)

plt.tight_layout()
plt.show()

# Summary table
summary = df.groupby('phq8_band', observed=True)['composite_score'].agg(
    n='count', mean='mean', std='std', median='median'
).round(3)
print('\nComposite score statistics per PHQ-8 severity band:')
print(summary.to_string())

In [ ]:
# ── Scatter: composite_score vs PHQ-8 ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left — coloured by risk label
risk_colors = df['risk_label'].map({0: '#5F9B6B', 1: '#C4715A'})
ax = axes[0]
ax.scatter(phq, comp, c=risk_colors, alpha=0.65, edgecolors='white', linewidths=0.4, s=55)
# regression line
m, b = np.polyfit(phq, comp, 1)
x_line = np.linspace(phq.min(), phq.max(), 200)
ax.plot(x_line, m * x_line + b, color='#2B2B2B', lw=1.5, ls='--', label=f'r={pearson_r:.2f}')
ax.axvline(10, color='grey', ls=':', lw=1, label='PHQ-8 = 10 cutoff')
ax.axhline(THRESHOLD, color='grey', ls='-.', lw=1, label=f'Risk threshold {THRESHOLD}')
ax.set_xlabel('PHQ-8 score'); ax.set_ylabel('Composite risk score')
ax.set_title('Composite score vs PHQ-8 (colour = risk label)')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#C4715A', label='High risk'),
    Patch(color='#5F9B6B', label='Low risk'),
    plt.Line2D([],[],color='#2B2B2B',ls='--', label=f'OLS  r={pearson_r:.2f}'),
    plt.Line2D([],[],color='grey',ls=':', label='PHQ-8 ≥ 10'),
    plt.Line2D([],[],color='grey',ls='-.', label=f'Composite ≥ {THRESHOLD}'),
], fontsize=8)

# Right — coloured by depression label (Depression_label from dataset)
dep_label_colors = df['Depression_label'].astype(int).map({0: '#4A7DBF', 1: '#E07B4A'})
ax2 = axes[1]
ax2.scatter(phq, comp, c=dep_label_colors, alpha=0.65, edgecolors='white', linewidths=0.4, s=55)
ax2.plot(x_line, m * x_line + b, color='#2B2B2B', lw=1.5, ls='--')
ax2.axvline(10, color='grey', ls=':', lw=1)
ax2.axhline(THRESHOLD, color='grey', ls='-.', lw=1)
ax2.set_xlabel('PHQ-8 score'); ax2.set_ylabel('Composite risk score')
ax2.set_title('Composite score vs PHQ-8 (colour = clinical depression label)')
ax2.legend(handles=[
    Patch(color='#E07B4A', label='Depressed (label=1)'),
    Patch(color='#4A7DBF', label='Not depressed (label=0)'),
], fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

phq  = df['Depression_severity'].astype(float)
comp = df['composite_score'].astype(float)

pearson_r,  pearson_p  = stats.pearsonr(phq, comp)
spearman_r, spearman_p = stats.spearmanr(phq, comp)

print('Correlation between composite_score and PHQ-8:')
print(f'  Pearson  r = {pearson_r:.4f}  (p = {pearson_p:.4e})')
print(f'  Spearman r = {spearman_r:.4f}  (p = {spearman_p:.4e})')
print()

# Also with raw PCL-C for reference
pclc = df['PTSD_severity'].astype(float)
pr2, pp2 = stats.pearsonr(pclc, comp)
print(f'Pearson  r (composite vs PCL-C) = {pr2:.4f}  (p = {pp2:.4e})')
print()
print('Note: composite_score is constructed from both PHQ-8 and PCL-C, so')
print('high correlation is expected — this section explores the PHQ-8 component specifically.')

## 3  Preprocessing

In [ ]:
# ── Feature selection: drop low-variance and high-missing ─────────────────────
from sklearn.feature_selection import VarianceThreshold

# Drop cols with >20% missing
drop_missing = missing[missing > 0.20].index.tolist()
feat_use = [c for c in feat_cols_all if c not in drop_missing]
print(f'Features after dropping high-missing: {len(feat_use)} / {len(feat_cols_all)}')

X_raw = df[feat_use].copy()
y     = df['composite_score'].values

# Impute remaining NaNs with column median
X_raw = X_raw.fillna(X_raw.median())

# Remove near-zero variance
vt = VarianceThreshold(threshold=1e-6)
vt.fit(X_raw)
feat_use = [f for f, keep in zip(feat_use, vt.get_support()) if keep]
X_raw = X_raw[feat_use]
print(f'Features after variance threshold: {len(feat_use)}')

feature_names = feat_use
print(f'\nFinal feature set: {len(feature_names)} features')

In [ ]:
# Scaled version (for linear models)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# CV strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_scores(model, X, y, cv=cv):
    """Return mean ± std for RMSE, MAE, R² using 5-fold CV."""
    res = cross_validate(
        model, X, y, cv=cv,
        scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
        return_train_score=False
    )
    return {
        'RMSE': (-res['test_neg_root_mean_squared_error']).mean(),
        'RMSE_std': (-res['test_neg_root_mean_squared_error']).std(),
        'MAE':  (-res['test_neg_mean_absolute_error']).mean(),
        'MAE_std': (-res['test_neg_mean_absolute_error']).std(),
        'R2':   res['test_r2'].mean(),
        'R2_std': res['test_r2'].std(),
    }

## 4  Baseline: Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf_cv = cv_scores(rf, X_raw, y)

print('Random Forest (5-fold CV):')
print(f"  RMSE : {rf_cv['RMSE']:.4f} ± {rf_cv['RMSE_std']:.4f}")
print(f"  MAE  : {rf_cv['MAE']:.4f} ± {rf_cv['MAE_std']:.4f}")
print(f"  R²   : {rf_cv['R2']:.4f} ± {rf_cv['R2_std']:.4f}")

## 5  Feature importance

In [ ]:
# Fit RF on full data for importance extraction
rf.fit(X_raw, y)

# ── Impurity-based (MDI) importance ──────────────────────────────────────────
imp_df = pd.DataFrame({
    'feature':    feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

TOP_N = 30
top_imp = imp_df.head(TOP_N)

plt.figure(figsize=(10, 8))
palette = ['#5F9B6B' if i < 5 else '#7DB08A' if i < 15 else '#A8C4AF'
           for i in range(TOP_N)]
plt.barh(top_imp['feature'][::-1], top_imp['importance'][::-1], color=palette[::-1])
plt.xlabel('MDI Importance')
plt.title(f'Random Forest — Top {TOP_N} Feature Importances (MDI)')
plt.tight_layout()
plt.show()

print(f'\nTop 15 features by MDI importance:')
print(imp_df.head(15).to_string(index=False))

In [ ]:
# ── Permutation importance (more reliable, model-agnostic) ────────────────────
perm = permutation_importance(rf, X_raw, y, n_repeats=20, random_state=42, n_jobs=-1)

perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm.importances_mean,
    'importance_std':  perm.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)

top_perm = perm_df[perm_df['importance_mean'] > 0].head(TOP_N)

plt.figure(figsize=(10, 8))
plt.barh(top_perm['feature'][::-1], top_perm['importance_mean'][::-1],
         xerr=top_perm['importance_std'][::-1],
         color='#4A7DBF', ecolor='#2B4F7A', capsize=3)
plt.xlabel('Permutation Importance (mean decrease in R²)')
plt.title(f'Permutation Importance — Top {min(TOP_N, len(top_perm))} Features')
plt.tight_layout()
plt.show()

print(f'\nTop 15 features by permutation importance:')
print(perm_df.head(15).to_string(index=False))

In [ ]:
# ── Feature category breakdown ────────────────────────────────────────────────
def categorise(name):
    if name.startswith('patient_mean'):
        return 'Language / Semantic'
    if 'mfcc' in name:
        return 'MFCC'
    if any(k in name for k in ['f0', 'pitch', 'f1_', 'f2_', 'f3_', 'f1_f2']):
        return 'Pitch / Formants'
    if any(k in name for k in ['pause', 'utterance', 'speech_rate', 'silence',
                                 'fragmented', 'filled_pause']):
        return 'Temporal / Rhythm'
    if any(k in name for k in ['intensity', 'hnr', 'shimmer', 'jitter',
                                 'breathiness', 'vocal_tension']):
        return 'Voice Quality'
    if any(k in name for k in ['spectral', 'hammarberg', 'alpha_ratio', 'band_']):
        return 'Spectral'
    return 'Other'

imp_df['category'] = imp_df['feature'].map(categorise)
cat_imp = imp_df.groupby('category')['importance'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
colors_cat = ['#5F9B6B','#4A7DBF','#C4715A','#A07DBF','#BF9A4A','#7DBFBF']
plt.bar(cat_imp.index, cat_imp.values, color=colors_cat[:len(cat_imp)])
plt.ylabel('Summed MDI Importance')
plt.title('Feature Importance by Category')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

print(cat_imp.round(4))

In [ ]:
# ── SHAP values (if available) ────────────────────────────────────────────────
if HAS_SHAP:
    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_raw)

    plt.figure()
    shap.summary_plot(shap_values, X_raw,
                      feature_names=feature_names,
                      max_display=20,
                      show=False)
    plt.title('SHAP Summary — Random Forest')
    plt.tight_layout()
    plt.show()
else:
    print('Install shap for SHAP plots: pip install shap')

## 6  Model comparison

In [ ]:
models = {
    'Ridge':    (Ridge(alpha=1.0),              X_scaled),
    'Lasso':    (Lasso(alpha=0.01),             X_scaled),
    'ElasticNet': (ElasticNet(alpha=0.01, l1_ratio=0.5), X_scaled),
    'SVR':      (SVR(kernel='rbf', C=1.0),      X_scaled),
    'GBRegressor': (GradientBoostingRegressor(n_estimators=200, random_state=42), X_raw),
    'RandomForest': (RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1), X_raw),
}

if HAS_XGB:
    models['XGBoost'] = (XGBRegressor(n_estimators=200, learning_rate=0.05,
                                       random_state=42, verbosity=0), X_raw)

results = []
for name, (model, X) in models.items():
    s = cv_scores(model, X, y)
    s['Model'] = name
    results.append(s)
    print(f"{name:<16}  RMSE={s['RMSE']:.4f}±{s['RMSE_std']:.4f}  "
          f"MAE={s['MAE']:.4f}  R²={s['R2']:.4f}")

results_df = pd.DataFrame(results).set_index('Model')[['RMSE','MAE','R2']]
print('\n', results_df.sort_values('RMSE').round(4))

In [ ]:
# ── Visualise model comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
results_sorted = pd.DataFrame(results).set_index('Model').sort_values('RMSE')

for ax, metric, color in zip(axes, ['RMSE','MAE','R2'],
                              ['#C4715A','#4A7DBF','#5F9B6B']):
    ax.barh(results_sorted.index[::-1], results_sorted[metric][::-1], color=color)
    ax.set_title(metric)
    ax.set_xlabel(metric)

plt.suptitle('Model Comparison (5-fold CV)', y=1.02)
plt.tight_layout()
plt.show()

## 7  Hyperparameter tuning (best model)

In [ ]:
# ── Tune Random Forest ────────────────────────────────────────────────────────
param_grid = {
    'n_estimators':      [200, 400],
    'max_depth':         [None, 8, 15],
    'min_samples_leaf':  [1, 3, 5],
    'max_features':      ['sqrt', 0.5],
}

gs = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
gs.fit(X_raw, y)

print('Best params:', gs.best_params_)
print(f'Best CV RMSE: {-gs.best_score_:.4f}')

In [ ]:
# Re-evaluate tuned model
best_rf = gs.best_estimator_
tuned = cv_scores(best_rf, X_raw, y)
print('Tuned RF  RMSE:', round(tuned['RMSE'],4), '  MAE:', round(tuned['MAE'],4), '  R²:', round(tuned['R2'],4))

# Refit and get importances
best_rf.fit(X_raw, y)
imp_tuned = pd.DataFrame({'feature': feature_names,
                           'importance': best_rf.feature_importances_})\
              .sort_values('importance', ascending=False)
print('\nTop 10 features (tuned RF):')
print(imp_tuned.head(10).to_string(index=False))

## 8  Classification variant (binary risk)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics  import classification_report, roc_auc_score
from sklearn.model_selection import cross_val_predict

y_cls = df['risk_label'].values
print('Class balance:', pd.Series(y_cls).value_counts().to_dict())

rfc = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1,
                              class_weight='balanced')

from sklearn.model_selection import StratifiedKFold
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
res_cls = cross_validate(
    rfc, X_raw, y_cls, cv=skf,
    scoring=['accuracy','f1_weighted','roc_auc'],
    return_train_score=False
)

print(f"Accuracy  : {res_cls['test_accuracy'].mean():.4f} ± {res_cls['test_accuracy'].std():.4f}")
print(f"F1 (wtd)  : {res_cls['test_f1_weighted'].mean():.4f} ± {res_cls['test_f1_weighted'].std():.4f}")
print(f"ROC-AUC   : {res_cls['test_roc_auc'].mean():.4f} ± {res_cls['test_roc_auc'].std():.4f}")

In [ ]:
# Detailed classification report (OOF predictions)
rfc.fit(X_raw, y_cls)
y_pred_cls = cross_val_predict(rfc, X_raw, y_cls, cv=skf)
print(classification_report(y_cls, y_pred_cls, target_names=['Low risk','High risk']))

# Classifier feature importances
rfc_imp = pd.DataFrame({'feature': feature_names,
                         'importance': rfc.feature_importances_})\
            .sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(rfc_imp['feature'].head(20)[::-1],
         rfc_imp['importance'].head(20)[::-1],
         color='#C4715A')
plt.xlabel('MDI Importance')
plt.title('RF Classifier — Top 20 Features for Binary Risk')
plt.tight_layout()
plt.show()

## 9  Export artefacts

In [ ]:
import json

# Feature importance tables
imp_df.to_csv('feature_importance_mdi.csv', index=False)
perm_df.to_csv('feature_importance_perm.csv', index=False)
results_df.to_csv('model_comparison.csv')

# Top features as JSON (useful for dashboard integration)
top_features = imp_df.head(20)[['feature','importance','category']].to_dict('records')
with open('top_features.json', 'w') as f:
    json.dump(top_features, f, indent=2)

print('Saved:')
print('  feature_importance_mdi.csv')
print('  feature_importance_perm.csv')
print('  model_comparison.csv')
print('  top_features.json')

---
### Summary

| What | Details |
|---|---|
| **Target** | `(PHQ-8/24 + PCL-C/68) / 2` — normalised composite distress score |
| **Features** | All acoustic + semantic features from `daic_features_v4.csv` |
| **Best model** | See model comparison table above |
| **Key features** | See `feature_importance_mdi.csv` and `top_features.json` |

**Next steps to try:**
- Feature selection: keep only top-K permutation-important features and rerun CV
- Train/dev/test split: use the `split` column from `detailed_labels.csv` for a held-out test set
- Separate regression targets for PHQ-8 and PCL-C independently
- Multi-task learning: predict PHQ and PCL-C simultaneously
- Add demographic features (age, gender) as covariates